# Jewelry Eval Notebook

Runs the eval pipeline from the repo modules and regenerates `eval/results/predictions.csv`,
confusion matrices, and `misclassified.csv`.

**Resumable by design**: if this notebook (or Colab) disconnects partway through sampling or
prediction, just run all cells again from the top. Neither step recomputes work that's already
saved to disk -- sampling reuses the existing `balanced_eval_sample.csv`, and the prediction loop
skips any `serial_number` already present in `predictions.csv` and picks up exactly where it left off.


## 1. Drive mount + paths (explicit, no auto-search)

In [ ]:
from pathlib import Path, PureWindowsPath
import os

DRIVE_MOUNT_POINT = Path("/content/drive")
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount(str(DRIVE_MOUNT_POINT))

# ---- Repo location (already saved on Drive -- no cloning needed) ----
PROJECT_ROOT = (DRIVE_MOUNT_POINT / "MyDrive" / "Jewelry-Search-Prod--feat-Inference-pipeline").resolve()
os.chdir(PROJECT_ROOT)

required_files = [
    PROJECT_ROOT / "config" / "config.yaml",
    PROJECT_ROOT / "engines" / "clip_engine.py",
    PROJECT_ROOT / "classifiers" / "category_classifier.py",
]
missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError(
        "PROJECT_ROOT is set but these expected files are missing:\n  "
        + "\n  ".join(missing)
        + f"\nCheck that {PROJECT_ROOT} is the correct folder and Drive is mounted."
    )

# ---- Data paths (metadata CSV + images live in a separate folder from the repo) ----
METADATA_CSV = DRIVE_MOUNT_POINT / "MyDrive" / "RFID-Project" / "rebuilt_labelled_metadata_new.csv"
IMAGES_ROOT = DRIVE_MOUNT_POINT / "MyDrive" / "RFID-Project"

for label, path in [("METADATA_CSV", METADATA_CSV), ("IMAGES_ROOT", IMAGES_ROOT)]:
    if not path.exists():
        raise FileNotFoundError(f"{label} does not exist: {path}. Check the path and that Drive is mounted.")

CONFIG_YAML = PROJECT_ROOT / "config" / "config.yaml"
PROMPTS_YAML = PROJECT_ROOT / "config" / "prompts.yaml"
OUTPUT_DIR = PROJECT_ROOT / "eval" / "results"
SAMPLED_EVAL_CSV = OUTPUT_DIR / "balanced_eval_sample.csv"
PREDICTIONS_CSV = OUTPUT_DIR / "predictions.csv"

BALANCED_SAMPLE_SIZE = 2000
RANDOM_STATE = 42

USE_SERVER_SIDE_BIREFNET = True
LIMIT = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT, METADATA_CSV, IMAGES_ROOT, OUTPUT_DIR


## 2. Install dependencies

In [ ]:
%pip install -q -r requirements.txt seaborn scikit-learn matplotlib ipywidgets


## 3. Imports

In [ ]:
import sys
import types
import yaml
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

sys.path.insert(0, str(PROJECT_ROOT))

from engines.clip_engine import CLIPEngine
from engines.dinov2_engine import DINOv2Engine
from engines.birefnet_engine import BiRefNetEngine
from preprocess.image_processor import ImageProcessor
from classifiers.category_classifier import CategoryClassifier
from classifiers.material_classifier import MaterialClassifier
from indexing.pipeline import IndexingPipeline

print(f"Project root: {PROJECT_ROOT}")
print(f"Writing results to: {OUTPUT_DIR}")


## 4. Config + model path overrides

In [ ]:
def load_config(path: Path):
    with open(path, "r") as f:
        raw = yaml.safe_load(f)
    return types.SimpleNamespace(**raw.get("inference", {}))


def load_prompts(path: Path):
    with open(path, "r") as f:
        return yaml.safe_load(f)


config = load_config(CONFIG_YAML)
prompts = load_prompts(PROMPTS_YAML)

# Model weight folders live under RFID-Project/models/ on Drive, not inside the repo.
MODELS_ROOT = IMAGES_ROOT / "models"

config.birefnet_model_path = str(MODELS_ROOT / "BiRefNet")
config.clip_model_path = str(MODELS_ROOT / "clip-ViT-L-14")

for label, path_str in [
    ("birefnet_model_path", config.birefnet_model_path),
    ("clip_model_path", config.clip_model_path),
]:
    if not Path(path_str).exists():
        raise FileNotFoundError(f"{label} does not exist: {path_str}. Check the Drive path.")

print(config)
print("Categories:", list(prompts["categories"].keys()))
print("Materials:", list(prompts["materials"].keys()))


## 5. Balanced eval sample (resumable)

If `balanced_eval_sample.csv` already exists in `OUTPUT_DIR`, it's loaded as-is and
**no re-sampling happens** -- this guarantees the eval set stays identical across
reruns/reconnects rather than drawing a new random sample each time.


In [ ]:
def resolve_image_path(labelled_path_value, original_filename_value=None, category_value=None) -> Path:
    raw = "" if pd.isna(labelled_path_value) else str(labelled_path_value).strip()
    filename = PureWindowsPath(raw).name if "\\" in raw else Path(raw).name
    path = Path(raw.replace("\\", "/"))

    candidates = []
    if path.is_absolute():
        candidates.append(path)
    else:
        candidates.extend([PROJECT_ROOT / path, IMAGES_ROOT / path])

    category = None if category_value is None or pd.isna(category_value) else str(category_value).strip()
    if category:
        candidates.extend([IMAGES_ROOT / category / path, IMAGES_ROOT / category / filename])

    if raw:
        candidates.append(IMAGES_ROOT / filename)

    if original_filename_value is not None and not pd.isna(original_filename_value):
        original_filename = str(original_filename_value).strip()
        candidates.append(IMAGES_ROOT / original_filename)
        if category:
            candidates.append(IMAGES_ROOT / category / original_filename)

    for candidate in candidates:
        if candidate.exists():
            return candidate

    return candidates[0] if candidates else IMAGES_ROOT / raw


def balanced_sample_by_category(df: pd.DataFrame, total: int, random_state: int) -> pd.DataFrame:
    categories = sorted(df["category"].dropna().unique())
    if not categories:
        raise ValueError("No categories found after cleaning metadata.")

    shuffled_groups = {
        category: group.sample(frac=1, random_state=random_state)
        for category, group in df.groupby("category", sort=True)
    }
    positions = {category: 0 for category in categories}
    selected_indices = []

    while len(selected_indices) < total:
        progressed = False
        for category in categories:
            group = shuffled_groups[category]
            pos = positions[category]
            if pos < len(group):
                selected_indices.append(group.index[pos])
                positions[category] += 1
                progressed = True
                if len(selected_indices) == total:
                    break
        if not progressed:
            break

    return df.loc[selected_indices].sample(frac=1, random_state=random_state).reset_index(drop=True)


def build_balanced_eval_sample():
    if SAMPLED_EVAL_CSV.exists():
        print(f"Eval sample already exists -- loading, not re-sampling: {SAMPLED_EVAL_CSV}")
        return pd.read_csv(SAMPLED_EVAL_CSV)

    metadata = pd.read_csv(METADATA_CSV)
    required = {"original_filename", "category", "material", "labelled_path"}
    missing_cols = required - set(metadata.columns)
    if missing_cols:
        raise ValueError(f"Metadata CSV is missing columns: {sorted(missing_cols)}")

    metadata = metadata.copy()
    metadata["category"] = metadata["category"].astype("string").str.strip()
    metadata = metadata[metadata["category"].notna() & (metadata["category"] != "")]
    metadata["material"] = metadata["material"].fillna("unknown").astype("string").str.strip()
    metadata.loc[metadata["material"] == "", "material"] = "unknown"

    metadata["image_path"] = metadata.apply(
        lambda row: str(resolve_image_path(row["labelled_path"], row["original_filename"], row["category"])),
        axis=1,
    )
    metadata["filename"] = metadata["image_path"].apply(lambda value: Path(value).name)
    metadata["serial_number"] = metadata["original_filename"].apply(lambda value: PureWindowsPath(str(value)).stem)
    duplicate_serials = metadata["serial_number"].duplicated(keep=False)
    metadata.loc[duplicate_serials, "serial_number"] = (
        metadata.loc[duplicate_serials, "serial_number"].astype(str) + "__row_" + metadata.loc[duplicate_serials].index.astype(str)
    )

    print(f"Metadata rows after cleaning: {len(metadata)}")
    display(metadata["category"].value_counts().rename("available_count").to_frame())

    sampled = balanced_sample_by_category(metadata, BALANCED_SAMPLE_SIZE, RANDOM_STATE)
    if LIMIT is not None:
        sampled = sampled.head(LIMIT).copy()

    sampled.to_csv(SAMPLED_EVAL_CSV, index=False)
    print(f"Saved new balanced eval sample: {SAMPLED_EVAL_CSV}")
    return sampled


gt = build_balanced_eval_sample()
print(f"Balanced eval rows: {len(gt)}")
display(gt.head())
display(gt["category"].value_counts().rename("sampled_count").to_frame())


## 6. Load engines + build pipeline

In [ ]:
print("Loading engines...")
birefnet = BiRefNetEngine(config) if USE_SERVER_SIDE_BIREFNET else None
clip = CLIPEngine(config)
dinov2 = DINOv2Engine(config)

print("Loading classifiers...")
cat_clf = CategoryClassifier(clip, prompts["categories"])
mat_clf = MaterialClassifier(clip, prompts["materials"])

# Do not pass BiRefNet into ImageProcessor in the current repo version.
processor = ImageProcessor()
pipeline = IndexingPipeline(processor, clip, dinov2, cat_clf, mat_clf)
print("Pipeline ready.")


## 7. Check for missing image files before running inference

In [ ]:
def image_path_for(row) -> Path:
    return Path(str(row["image_path"]))


missing = []
for _, row in gt.iterrows():
    path = image_path_for(row)
    if not path.exists():
        missing.append(str(path))

print(f"Missing images: {len(missing)}")
if missing:
    print("First missing paths:")
    for path in missing[:10]:
        print("  ", path)


## 8. Run inference (resumable, checkpointed every 10 rows)

- If `predictions.csv` already exists, it's loaded first and any `serial_number` already
  present there is **skipped** -- so a rerun after a disconnect only processes what's left.
- Results are saved to disk every 10 rows (atomic write: temp file + `os.replace`), not just
  at the very end -- a crash mid-run loses at most the current partial batch, not everything.


In [ ]:
SAVE_EVERY = 10

def atomic_save_predictions(df: pd.DataFrame):
    tmp_path = PREDICTIONS_CSV.with_suffix(".csv.tmp")
    df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, PREDICTIONS_CSV)


if PREDICTIONS_CSV.exists():
    preds_df = pd.read_csv(PREDICTIONS_CSV)
    print(f"Existing predictions found -- resuming. Already processed: {len(preds_df)}")
else:
    preds_df = pd.DataFrame(columns=[
        "serial_number", "predicted_category", "category_confidence",
        "predicted_material", "material_confidence", "error",
    ])
    print("No existing predictions.csv -- starting fresh.")

already_done = set(preds_df["serial_number"].astype(str))
gt_reset = gt.reset_index(drop=True)
remaining = gt_reset[~gt_reset["serial_number"].astype(str).isin(already_done)]
print(f"Remaining to process: {len(remaining)}/{len(gt_reset)}")

new_predictions = []
since_last_save = 0

for i, row in remaining.iterrows():
    img_path = image_path_for(row)
    if not img_path.exists():
        print(f"MISSING: {img_path}")
        continue

    try:
        opened_image = Image.open(img_path)
        if USE_SERVER_SIDE_BIREFNET:
            image_for_pipeline = birefnet.get_rgba(opened_image.convert("RGB"))
        else:
            image_for_pipeline = opened_image.convert("RGBA")

        record = pipeline.process(
            serial_number=str(row["serial_number"]),
            image=image_for_pipeline,
            tenant_id="eval",
        )
        new_predictions.append({
            "serial_number": row["serial_number"],
            "predicted_category": record.category,
            "category_confidence": record.category_confidence,
            "predicted_material": record.material,
            "material_confidence": record.material_confidence,
            "error": record.error,
        })
    except Exception as exc:
        print(f"ERROR {row['serial_number']}: {exc}")
        new_predictions.append({
            "serial_number": row["serial_number"],
            "predicted_category": None,
            "category_confidence": None,
            "predicted_material": None,
            "material_confidence": None,
            "error": str(exc),
        })

    since_last_save += 1
    total_processed = len(preds_df) + len(new_predictions)

    if since_last_save >= SAVE_EVERY or total_processed == len(gt_reset):
        preds_df = pd.concat([preds_df, pd.DataFrame(new_predictions)], ignore_index=True)
        new_predictions = []
        since_last_save = 0
        atomic_save_predictions(preds_df)
        print(f"Checkpoint saved. Processed {total_processed}/{len(gt_reset)}")

if new_predictions:
    preds_df = pd.concat([preds_df, pd.DataFrame(new_predictions)], ignore_index=True)
    atomic_save_predictions(preds_df)

print(f"Saved: {PREDICTIONS_CSV}")
display(preds_df.head())


## 9. Metrics

In [ ]:
df = gt.merge(preds_df, on="serial_number")
df = df[df["error"].isna()].copy()
print(f"Successfully processed: {len(df)}/{len(gt)}")

print("\n=== CATEGORY HEAD ===")
cat_acc = accuracy_score(df["category"], df["predicted_category"])
print(f"Accuracy: {cat_acc:.1%}")
print(classification_report(df["category"], df["predicted_category"], zero_division=0))

mat_df = df[df["material"] != "unknown"].copy()
if len(mat_df) > 0:
    print("\n=== MATERIAL HEAD ===")
    mat_acc = accuracy_score(mat_df["material"], mat_df["predicted_material"])
    print(f"Accuracy: {mat_acc:.1%}")
    print(classification_report(mat_df["material"], mat_df["predicted_material"], zero_division=0))
else:
    mat_acc = None
    print("\nNo non-unknown material labels found; skipping material metrics.")


## 10. Confusion matrices

In [ ]:
def save_confusion_matrix(y_true, y_pred, labels, title, cmap, path: Path, figsize):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=figsize)
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=labels, yticklabels=labels, cmap=cmap)
    plt.title(title)
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()
    print(f"Saved: {path}")


cat_labels = sorted(df["category"].unique())
save_confusion_matrix(
    df["category"],
    df["predicted_category"],
    cat_labels,
    f"Category Confusion Matrix (Accuracy: {cat_acc:.1%})",
    "Reds",
    OUTPUT_DIR / "confusion_matrix_category.png",
    (14, 10),
)

if len(mat_df) > 0:
    mat_labels = sorted(mat_df["material"].unique())
    save_confusion_matrix(
        mat_df["material"],
        mat_df["predicted_material"],
        mat_labels,
        f"Material Confusion Matrix (Accuracy: {mat_acc:.1%})",
        "Blues",
        OUTPUT_DIR / "confusion_matrix_material.png",
        (10, 8),
    )


## 11. Misclassified rows

In [ ]:
misclassified = df[
    (df["category"] != df["predicted_category"]) |
    (df["material"] != df["predicted_material"])
][[
    "serial_number",
    "category",
    "predicted_category",
    "category_confidence",
    "material",
    "predicted_material",
    "material_confidence",
]]

misclassified_path = OUTPUT_DIR / "misclassified.csv"
misclassified.to_csv(misclassified_path, index=False)
print(f"Misclassified: {len(misclassified)}/{len(df)}")
print(f"Saved: {misclassified_path}")
display(misclassified.head(20))


## Notes

- `PROJECT_ROOT` points directly at the Drive-hosted repo folder; no git clone step needed.
- `config.birefnet_model_path` / `config.clip_model_path` are overridden to
  `RFID-Project/models/...` on Drive, since those weight folders live outside the repo.
- **Resumability**: both the balanced-sampling step (Cell 5) and the inference loop (Cell 8)
  check for existing output files first and only do the remaining work -- safe to rerun the
  whole notebook after any disconnect without losing progress or redrawing a different sample.
- If DINOv2 is not cached, `torch.hub.load("facebookresearch/dinov2", ...)` needs network
  access the first time.
